# PopDAM Bulk Vision Tagging Server (Google Colab)

Runs an OpenAI-compatible, continuously batched **vLLM** server for PopDAM bulk image tagging (`Qwen/Qwen3-VL-32B-Instruct`) on a high-VRAM Colab GPU (G4 96 GB / A100 80 GB / H100 80 GB) behind a temporary authenticated HTTPS tunnel.

- **Zero database credentials in Colab:** PopDAM's Railway worker keeps all Supabase access, tagging rules, character binding, and cursor resume.
- **Instant shutdown by default:** When PopDAM finishes the last file in a bulk tagging or bake-off job, the worker sends an authenticated `POST /popdam/shutdown` signal and this notebook immediately unassigns the Colab GPU within seconds.
- **Idle safety net:** If the worker disconnects or crashes before signaling completion, the notebook still auto-unassigns after `IDLE_SHUTDOWN_MINUTES` so compute units are never wasted.

In [ ]:
# === 1. Configuration ===
MODEL_ID = "Qwen/Qwen3-VL-32B-Instruct"
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTILIZATION = 0.90
IDLE_SHUTDOWN_MINUTES = 20  # Backup safety net if the worker disconnects without signaling shutdown


In [ ]:
# === 2. Install vLLM + cloudflared, launch server + shutdown proxy + tunnel ===
import hmac, json, os, re, secrets, subprocess, threading, time, urllib.error, urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

print("=== GPU ===")
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"]).decode().strip())

print("\n=== Installing vLLM and cloudflared ===")
subprocess.check_call(["pip", "install", "-q", "-U", "vllm", "qwen-vl-utils"])
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.check_call([
        "curl", "-fsSL",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-o", "/usr/local/bin/cloudflared",
    ])
    os.chmod("/usr/local/bin/cloudflared", 0o755)

API_KEY = "colab_" + secrets.token_urlsafe(32)
VLLM_PORT = 8001
PROXY_PORT = 8000

vllm_log = open("/tmp/vllm.log", "w")
vllm_proc = subprocess.Popen([
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--host", "127.0.0.1",
    "--port", str(VLLM_PORT),
    "--api-key", API_KEY,
    "--dtype", "bfloat16",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--limit-mm-per-prompt", "image=1",
], stdout=vllm_log, stderr=subprocess.STDOUT)

print(f"\n=== Starting vLLM for {MODEL_ID} (takes ~2-4 min on first download) ===")
ready = False
for _ in range(360):
    if vllm_proc.poll() is not None:
        raise RuntimeError("vLLM exited early; check /tmp/vllm.log")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{VLLM_PORT}/health", timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(2)
if not ready:
    raise RuntimeError("Timed out waiting for vLLM /health")

shutdown_event = threading.Event()
shutdown_reason = {"reason": ""}
last_activity = {"ts": time.time()}

def authorized(auth_header):
    expected = f"Bearer {API_KEY}"
    return isinstance(auth_header, str) and hmac.compare_digest(auth_header.strip(), expected)

class PopdamProxyHandler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    def log_message(self, format, *args):
        pass

    def _forward(self):
        length = int(self.headers.get("Content-Length", "0") or "0")
        body = self.rfile.read(length) if length > 0 else None

        if self.command == "POST" and self.path.rstrip("/") == "/popdam/shutdown":
            if not authorized(self.headers.get("Authorization")):
                payload = b'{"error":"unauthorized"}'
                self.send_response(401)
                self.send_header("Content-Type", "application/json")
                self.send_header("Content-Length", str(len(payload)))
                self.end_headers()
                self.wfile.write(payload)
                return
            try:
                parsed = json.loads(body.decode("utf-8")) if body else {}
                shutdown_reason["reason"] = str(parsed.get("reason", "worker_signal"))
            except Exception:
                shutdown_reason["reason"] = "worker_signal"
            payload = json.dumps({"ok": True, "shutting_down": True}).encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(payload)))
            self.end_headers()
            self.wfile.write(payload)
            shutdown_event.set()
            return

        if self.path.startswith("/v1/") and authorized(self.headers.get("Authorization")):
            last_activity["ts"] = time.time()

        headers = {
            k: v for k, v in self.headers.items()
            if k.lower() not in ("host", "connection", "content-length", "transfer-encoding")
        }
        req = urllib.request.Request(
            f"http://127.0.0.1:{VLLM_PORT}{self.path}",
            data=body,
            headers=headers,
            method=self.command,
        )
        try:
            with urllib.request.urlopen(req, timeout=300) as resp:
                resp_body = resp.read()
                status = resp.status
                ctype = resp.headers.get("Content-Type", "application/json")
        except urllib.error.HTTPError as err:
            resp_body = err.read()
            status = err.code
            ctype = err.headers.get("Content-Type", "application/json")
        except Exception as err:
            resp_body = json.dumps({"error": str(err)}).encode("utf-8")
            status = 502
            ctype = "application/json"

        self.send_response(status)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(resp_body)))
        self.end_headers()
        self.wfile.write(resp_body)

    do_GET = _forward
    do_POST = _forward

proxy = ThreadingHTTPServer(("127.0.0.1", PROXY_PORT), PopdamProxyHandler)
proxy_thread = threading.Thread(target=proxy.serve_forever, daemon=True)
proxy_thread.start()

cf_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PROXY_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
tunnel_url = None
for line in cf_proc.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        break
if not tunnel_url:
    raise RuntimeError("Failed to obtain Cloudflare tunnel URL")

print("\n=== READY FOR POPDAM ===")
print("Set admin_config.COLAB_VISION_CONFIG to:")
print(json.dumps({"base_url": tunnel_url, "api_key": API_KEY, "auto_shutdown": True}, indent=2))
print(f"\nPopDAM model ID: colab-direct/{MODEL_ID}")
print(f"Instant shutdown on job completion: ENABLED (backup idle safety net: {IDLE_SHUTDOWN_MINUTES} min).")

try:
    while not shutdown_event.wait(timeout=10):
        if vllm_proc.poll() is not None or cf_proc.poll() is not None:
            print("Server or tunnel stopped; shutting down.")
            break
        idle_min = (time.time() - last_activity["ts"]) / 60.0
        if idle_min >= IDLE_SHUTDOWN_MINUTES:
            print(f"Idle safety net triggered ({idle_min:.1f} min >= {IDLE_SHUTDOWN_MINUTES} min). Unassigning Colab runtime...")
            break
    if shutdown_event.is_set():
        print(f"Instant shutdown received from PopDAM ({shutdown_reason['reason']}). Unassigning Colab runtime now...")
        time.sleep(1)
finally:
    proxy.shutdown()
    cf_proc.terminate()
    vllm_proc.terminate()
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass
